In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from datetime import timedelta
import logging
import mysql

In [ ]:
# Prepare the DataFrame for the investment window analysis by ensuring proper data types and sorting.
def prepare_data(df, asset_class=None, start=pd.Timestamp.now(), end=pd.DateOffset(months=3)):
    """
    Prepare and clean the DataFrame for analysis.
    Args:
        df: DataFrame with 'TransactionDate', 'TransactionClass', and 'Available' columns
        asset_class: String to filter TransactionClass column (e.g., 'US Agencies')
        start_date_parameter: Starting date for analysis, defaults to today
        end_date_parameter: Ending date for analysis, defaults to 3 months from today
    Returns:
        DataFrame: Cleaned and sorted DataFrame
    """
    # Define start and end dates
    start_date = pd.Timestamp(start).normalize()
    end_date = pd.Timestamp(end).normalize()

    # 1. Filter the data within the date range
    data = df[(df['TransactionDate'] >= start_date) & (df['TransactionDate'] <= end_date)].copy()

    # 2. Filter by asset class if specified
    if asset_class is not None:
        # Check if asset_class exists in TransactionClass column
        if asset_class not in df['TransactionClass'].unique():
            raise ValueError(f"Asset class '{asset_class}' not found in data")
        data = data[data['TransactionClass'] == asset_class].copy()
        if data.empty:
            raise ValueError(f"No data found for asset class: {asset_class}")

    # Set the TransactionDate to datetime
    data['TransactionDate'] = pd.to_datetime(data['TransactionDate'])

    # Sort
    data = data.sort_values('TransactionDate').reset_index(drop=True)
    data['TransactionClass'] = asset_class if asset_class else 'Not Specified'
    return data[['TransactionDate', 'Available','TransactionClass']]

In [ ]:
def lookback_to_update_intervals(df, date, value):
    # find the date in the chronologically sorted data in the df.`TransactionDate` column
    # iterate backwards over the df from that date
    # return the earliest date where df.`Available` is greater or equal to the value passed



    # Filter df to only include dates on or before the given date
    df_filtered = df[df['TransactionDate'] <= date].copy()

    # Available column is already float64, no conversion needed

    # Sort ASCENDING to go from oldest to newest
    df_filtered = df_filtered.sort_values('TransactionDate', ascending=False)

    # print("Available: ", value)
    # print(df_filtered)

    # Iterate from oldest to newest to find the earliest date where condition is met
    for _, row in df_filtered.iterrows():
        if row['Available'] < value and 'previous_date' in locals():
            return previous_date
        previous_date = row['TransactionDate']
            # return row['TransactionDate']
    # If no date found
    return None

In [ ]:
"""
Determine investment windows where the available balance exceeds a threshold and track the duration of these windows.
df: DataFrame with 'TransactionDate' and 'Available' columns
threshold = 1,000,000 - minimum balance to consider
Returns the investment windows as a DataFrame with columns:
    'start_date': Start date of the investment window
    'end_date': End date of the investment window (None if ongoing)
    'amount': The balance amount during the window
    'duration': Duration of the window in days
"""
def find_window_intervals(df, threshold=1_000_000):
    # dictionary format
    intervals = []
    intervals.append({
        'start_date': df.iloc[0]['TransactionDate'],
        'end_date': None,
        'amount': df.iloc[0]['Available'],
        'duration': 0
    })
    # Initialize
    current_amount = df.iloc[0]['Available']
    # Iterate through the source dataframe
    for i in range(1, len(df)):
        current_amount = df.iloc[i]['Available']
        prior_day_amount = df.iloc[i-1]['Available']
        # Start a new interval
        if current_amount != prior_day_amount and current_amount > threshold and i != len(df) - 1:
            intervals.append({
                'start_date': df.iloc[i]['TransactionDate'],
                'end_date': None,
                'amount': df.iloc[i]['Available'],
                'duration': 0
            })
        # Update open intervals
        for interval in intervals:
            if interval['end_date'] is None:
                # increment
                if current_amount >= interval['amount']:
                    interval['duration'] += 1
                else:
                    # finalize
                    interval['end_date'] = df.iloc[i]['TransactionDate']
                    # earliest_available_date = lookback_to_update_intervals(df, df.iloc[i]['TransactionDate'], df.iloc[i]['Available'])
                    # if earliest_available_date:
                    #     interval['start_date'] = earliest_available_date

    # Dataframe
    df_intervals = pd.DataFrame(intervals)
    df_intervals['start_date'] = df_intervals['start_date'].dt.date
    df_intervals['end_date'] = df_intervals['end_date'].dt.date
    df_intervals['amount'] = df_intervals['amount'].round(2)
    df_intervals['change'] = (df_intervals['amount'] - df_intervals['amount'].shift(1).fillna(0)).round(2)
    # Calculate amount change
    # df_intervals['difference'] = df_intervals['amount'].diff()
    # Set the first row AmountChange to equal the Amount
    # if len(df_intervals['amount'] = df_intervals['amount'].round(2)
    #     df_intervals.loc[0, 'Difference'] = df_intervals.loc[0, 'Amount']

    return df_intervals

In [ ]:
# MAIN
#     analyze_balance_data() for running the asset class balances processing

# In prod we'll be looping over each asset class and process the investment windows one by one
asset_classes = ['Certificate of Deposit', 'Mutual Fund', 'Commercial Paper', 'Money Market', 'US Treasuries', 'US Agencies']
asset_index = 5
# Load and print the data
running_balances = pd.read_pickle('running_balances.pkl')
data = prepare_data(running_balances, asset_classes[asset_index], '2025-09-04', '2025-10-31')

# Test
test = lookback_to_update_intervals(data, '2025-09-08', 23637591.81)
print("Lookback test result:", test)

with pd.option_context('display.max_columns', None,
                       'display.max_rows', None,
                       'display.width', None,
                       'display.expand_frame_repr', False):
    pd.options.display.float_format = '${:,.2f}'.format
display(data.head(30))

data.info()

intervals = find_window_intervals(data)
# Debug display
print("Investment windows:", len(intervals))
from IPython.display import display
display(intervals)